# Writing LTN formulas with TPTP

This companion to the [LTN basics](README.md) shows how to write formulas as text while keeping the same LTN constants, predicates and fuzzy operators. Read the tutorials on [grounding](1-grounding_non_logical_symbols.ipynb), [connectives and quantifiers](2-grounding_connectives.ipynb), [operators and gradients](2b-operators_and_gradients.ipynb), [learning](3-knowledgebase_and_learning.ipynb), and [reasoning](4-reasoning.ipynb) first.

We will supply data and a custom classifier, compare native and parsed formulas, train the classifier, and search for a counterexample in a separate problem. The parser reads TPTP's first-order (`fof`) and clause (`cnf`) languages. **Evaluation still uses fuzzy LTN semantics over a finite grounding domain.** A high satisfaction value is not a classical theorem-proving certificate.

Install LTN with its optional parser dependency before running this notebook:

```sh
pip install "ltn[parser]"
```

When working from a repository checkout, use `pip install -e ".[parser]"` instead. See the [parser README](../ltn/parser/README.md) for file loading, supported syntax and errors. All executable cells below are ordinary Python.

In [1]:
import tensorflow as tf
import ltn
from ltn.parser import Grounding, tptp

print("TensorFlow:", tf.__version__)

TensorFlow: 2.15.1


## A small classification problem

Two named points have known labels: `p(a)` is true and `p(b)` is false. A further rule says that a point known to be positive should satisfy `p`. We also retain `p(b)` as a conjecture to query; it is deliberately inconsistent with the negative label, so it should receive a low truth value after training.

Each TPTP statement has a name, a role and a formula. Predicate and constant names start with lowercase letters; variables such as `X` start with uppercase letters. Statements end in a period. Queries passed to `evaluate()` contain only a formula, without the enclosing `fof(...)` or final period. All query variables must be quantified.

In [2]:
problem = tptp.parse("""
    fof(labels, axiom, p(a) & ~p(b)).
    fof(rule, axiom, ! [X] : (known_positive(X) => p(X))).
    fof(question, conjecture, p(b)).
""")

grounding = Grounding(problem, embedding_size=1)
grounding.constants["a"] = ltn.Constant([1.0], trainable=False)
grounding.constants["b"] = ltn.Constant([-1.0], trainable=False)

print("Domain names:", grounding.domain)
assert set(grounding.domain) == {"a", "b"}
assert len(grounding.individuals()) == 2

Domain names: ['a', 'b']


`Grounding` initially supplies trainable embeddings for constants and MLPs for predicates and functions. Here we replace both constants with fixed one-dimensional data **before training**. The names `a` and `b` do not themselves specify these feature values.

Every quantified variable uses the same domain: here the two points `[1.0]` and `[-1.0]`. Constants used only in conjectures also belong to the problem and can enter this domain. Replacing a constant preserves its existing domain name; adding a new constant to `grounding.constants` alone does not add it to `grounding.domain`.

We now supply a fixed predicate `known_positive` and a learned predicate `p`. `Predicate.FromLogits` expects its logits model to accept a **list of tensors**, even for a unary predicate. Its native call is therefore `P([x])`. The parser preserves this convention. The ordinary unary Lambda predicate instead receives a tensor and is called as `KnownPositive(x)`.

In [3]:
class Logits(tf.keras.Model):
    """A unary classifier whose input follows FromLogits' list convention."""

    def __init__(self) -> None:
        super().__init__(name="tutorial_logits")
        self.dense = tf.keras.layers.Dense(
            1, kernel_initializer="zeros", bias_initializer="zeros"
        )

    def call(self, inputs: list[tf.Tensor]) -> tf.Tensor:
        return self.dense(inputs[0])


grounding.predicates["known_positive"] = ltn.Predicate.Lambda(
    lambda x: tf.cast(x[:, 0] > 0, tf.float32)
)
grounding.predicates["p"] = ltn.Predicate.FromLogits(
    Logits(), activation_function="sigmoid", with_class_indexing=False
)

P = grounding.predicates["p"]
KnownPositive = grounding.predicates["known_positive"]

## Compare the native and textual formulas

The notation changes; the grounded objects do not. We reuse the same predicates, constants and operator objects in both forms.

| Native LTN expression | TPTP formula |
| --- | --- |
| `P([a])` | `p(a)` |
| `Not(P([b]))` | `~p(b)` |
| `And(P([a]), Not(P([b])))` | `(p(a) & ~p(b))` |
| `Forall(x, Implies(KnownPositive(x), P([x])))` | `! [X] : (known_positive(X) => p(X))` |

TPTP also uses `|` for disjunction, `<=>` for equivalence and `? [X] : ...` for existential quantification. Parenthesize compound formulas to make their scope explicit.

The default grounding uses the stable product configuration: product conjunction, probabilistic-sum disjunction and Reichenbach implication. Universal quantification uses p-mean error with `p=2`, existential quantification uses p-mean with `p=6`, and formula aggregation uses p-mean error with `p=2`. These choices affect both truth values and learning, as discussed in the operators tutorial.

In [4]:
Not = grounding.Not
And = grounding.connectives["&"]
Implies = grounding.connectives["=>"]
Forall = grounding.quantifiers["!"]
SatAgg = grounding.aggregator

rule_text = "! [X] : (known_positive(X) => p(X))"
# Parse once when a formula will be evaluated repeatedly.
rule = problem.parse_formula(rule_text)


def native_rule() -> ltn.core.Formula:
    """Evaluate the textual rule using native LTN calls."""
    # These data constants are fixed, so stacking them needs no GradientTape.
    x = ltn.Variable(
        "X", tf.stack([grounding.constants[name].tensor for name in grounding.domain])
    )
    return Forall(x, Implies(KnownPositive(x), P([x])))


def native_satisfaction() -> tf.Tensor:
    """Aggregate the same two axioms as Grounding.sat()."""
    a, b = grounding.constants["a"], grounding.constants["b"]
    return SatAgg([And(P([a]), Not(P([b]))), native_rule()]).tensor


parsed_rule = grounding.evaluate(rule)
tf.debugging.assert_near(parsed_rule.tensor, native_rule().tensor, atol=1e-6)
tf.debugging.assert_near(grounding.sat(), native_satisfaction(), atol=1e-6)
assert parsed_rule.free_vars == []

before = float(grounding.sat())
print(f"Initial rule, native: {float(native_rule().tensor):.6f}")
print(f"Initial rule, text:   {float(parsed_rule.tensor):.6f}")
print(f"Initial satisfaction: {before:.6f}")

Initial rule, native: 0.646376
Initial rule, text:   0.646376
Initial satisfaction: 0.413673


The conjunction of the two labels and the universally quantified rule are two separate axioms. `sat()` aggregates their truth values. This problem has no interpreted literals, so it contributes no implicit distinctness constraint.

For trainable constants in a native training loop, use `Variable.from_constants(..., tape=tape)` inside the tape rather than stacking once outside it. `Grounding` constructs its variable domains during evaluation, so `train()` records the required gradients automatically.

## Train and query the same classifier

`train()` maximizes the satisfaction of the selected axioms. The zero-initialized classifier makes this small example reproducible; randomly initialized models can produce different values. After training, we query individual points and compare both formula implementations again. This also verifies that the native calls see the weights learned through the parser.

In [5]:
grounding.train(200)

after = float(grounding.sat())
positive_truth = float(grounding.evaluate("p(a)").tensor)
negative_truth = float(grounding.evaluate("p(b)").tensor)
parsed_rule = grounding.evaluate(rule)

print(f"Final satisfaction: {after:.6f}")
print(f"p(a): {positive_truth:.6f}")
print(f"p(b): {negative_truth:.6f}")
print(f"Rule, native: {float(native_rule().tensor):.6f}")
print(f"Rule, text:   {float(parsed_rule.tensor):.6f}")

assert after > before
assert positive_truth > 0.8
assert negative_truth < 0.2
tf.debugging.assert_near(parsed_rule.tensor, native_rule().tensor, atol=1e-6)
tf.debugging.assert_near(grounding.sat(), native_satisfaction(), atol=1e-6)

Final satisfaction: 0.816894
p(a): 0.884913
p(b): 0.147607
Rule, native: 0.918496
Rule, text:   0.918496


### Axioms and conjectures have different roles

The default training selection includes `axiom` and other premise roles such as `hypothesis`, `definition` and `assumption`. It excludes both `conjecture` and `negated_conjecture`. A conjecture is available for evaluation without becoming a training target.

The next cell deliberately includes every statement in the satisfaction calculation. The conjecture `p(b)` conflicts with our negative label, so this score is lower. Pass `statements=problem.statements` to `train()` only when you intend to optimize every statement. This distinction matters for CNF theorem-prover problems, where the negated conjecture is often already present as clauses.

In [6]:
print("Default axioms:", [s.name for s in problem.with_role("axiom")])
print("Conjectures:", [s.name for s in problem.with_role("conjecture")])

all_statements_sat = float(grounding.sat(statements=problem.statements))
print(f"Default satisfaction:        {float(grounding.sat()):.6f}")
print(f"Including the conjecture:    {all_statements_sat:.6f}")

assert len(problem.with_role("axiom")) == 2
assert len(problem.with_role("conjecture")) == 1
assert all_statements_sat < after

Default axioms: ['labels', 'rule']
Conjectures: ['question']
Default satisfaction:        0.816894
Including the conjecture:    0.485654


`train()` calls custom models in training mode; `evaluate()`, `sat()` and refutation use inference mode. For a custom training loop, call `sat(tape=tape, training=True)` inside a `GradientTape`, then collect `grounding.trainable_variables` after the forward pass so newly built model weights are included.

Each `train()` call creates a new graph and, unless supplied, a new optimizer. For repeated calls with the same objects and formulas, `reuse_graph=True` keeps both. Cached graphs see assignments to existing weights, but can silently retain old predicate objects, constants, domains, connectives or quantifier settings. Configure the grounding first; after structural changes, use a fresh optimizer to rebuild the cached graph, or call without graph reuse.

## Equality and the finite domain

For ordinary terms, the default equality predicate is smooth similarity:

$$\operatorname{Eq}(u,v) = \exp(-\|u-v\|^2).$$

It is not a crisp identity test. Our fixed points are distance 2 apart, so `a = b` has truth value `exp(-4)`.

TPTP numeric literals and distinct objects, such as `"Apple"`, receive exact value comparisons when both sides of an equality are interpreted literals. Such comparisons need no grounding and do not extend the domain. Outside these comparisons, the default numeric-literal embeddings encode identity, not magnitude or numeric features; a token such as `2` does not automatically supply the tensor `[2.0]` to a predicate. Replace the constants and models when grounding numeric data. Defined arithmetic symbols such as `$less` and `$sum` are unsupported.

In [7]:
domain_before = list(grounding.domain)
soft_equality = grounding.evaluate("a = b").tensor
same_number = grounding.evaluate("1/2 = 0.5").tensor
different_objects = grounding.evaluate('"Apple" != "Microsoft"').tensor

print(f"a = b: {float(soft_equality):.6f}")
print(f"1/2 = 0.5: {float(same_number):.1f}")
print(f'"Apple" != "Microsoft": {float(different_objects):.1f}')
print("Domain names after the queries:", grounding.domain)

tf.debugging.assert_near(soft_equality, tf.exp(tf.constant(-4.0)), atol=1e-7)
assert float(same_number) == 1.0
assert float(different_objects) == 1.0
assert grounding.domain == domain_before

a = b: 0.018316
1/2 = 0.5: 1.0
"Apple" != "Microsoft": 1.0
Domain names after the queries: ['a', 'b']


If distinct interpreted values occur in the problem itself, `sat()` also includes an implicit constraint keeping their embeddings apart, even if they appear only in conjectures. Equal numeric spellings, such as `1` and `1.0`, share one grounding. See the [parser README](../ltn/parser/README.md) before replacing these groundings.

## Search for a counterexample in a fresh grounding

Training a knowledge base and measuring one query does not establish logical consequence. As in the reasoning tutorial, we can instead search for a grounding that satisfies the premises above a threshold `q` while making the query less true than `q`.

Consider `a | b` as the only axiom and `a` as the conjecture. Here `a` and `b` are **propositions**, with no arguments, rather than data constants. We create a separate, fresh grounding: refutation changes its parameters, and a previously trained, saturated classifier may have poor search gradients.

In [8]:
refutation_problem = tptp.parse("""
    fof(choice, axiom, a | b).
    fof(question, conjecture, a).
""")
counterexample = Grounding(refutation_problem)
q = 0.95
sat_kb, truth_query = counterexample.refute("a", q=q, steps=500)
found_counterexample = sat_kb >= q and truth_query < q

print(f"Axiom satisfaction: {sat_kb:.6f}")
print(f"Conjecture truth:   {truth_query:.6f}")
print("Counterexample found:", found_counterexample)
assert found_counterexample

Axiom satisfaction: 0.999800
Conjecture truth:   0.000000
Counterexample found: True


The returned pair is the final axiom satisfaction and query truth, not a proof status. In this example, satisfying `a | b` while falsifying `a` provides a counterexample for the configured LTN problem and threshold.

**If the search does not find a counterexample, the result is inconclusive.** Refutation uses local gradient optimization; a failed search can reflect an unsuitable initialization, limited model capacity or difficult optimization. It does not prove the conjecture, and a high query value in a single trained grounding does not prove it either.

## When to keep the native API

The parser is useful when formulas already exist as TPTP text or when you want to separate textual rules from their neural groundings. It does not express every feature of the native API.

| Native feature | Parser behavior |
| --- | --- |
| Separate domains for points, labels or other variable types | Every quantified variable uses one shared list of individuals. `evaluate(individuals=...)` can override that list for an evaluation, but does not assign a different domain to each variable. |
| Paired samples and labels through `ltn.diag` | No direct textual equivalent. Multiple variables use Cartesian products. |
| Guarded quantifiers with `mask=` | No direct textual equivalent. A fuzzy implication generally changes the aggregation and is not an equivalent mask. |
| Per-call quantifier `p=` or `stable=` | No textual per-call override. Replace the objects in `grounding.quantifiers` to set shared behavior before training. |
| Free-variable queries and per-variable minibatch domains | Textual queries must be closed; the built-in training loop uses the grounding's domain. Keep the native API for these data flows. |

For example, the learning tutorial uses separate point variables and a label variable. Translating its formulas into text without changing the domain setup would make labels range over points as well, and would not reproduce that experiment. The earlier notebooks remain the guide for such native workflows.

A formula with `v` variables over `n` individuals can build a tensor with `n**v` entries. Start with a small domain. Function applications in quantified domains are included only up to the chosen `herbrand_depth`; this is a finite expansion and can itself grow quickly. Quantified variables that occur in the body need a nonempty domain; for a problem without constants, one option is `Grounding(problem, extra_individuals=1)`. Propositional formulas, such as the refutation example above, do not need individuals.

Use the [parser README](../ltn/parser/README.md) for `parse_file`, TPTP includes, unsupported languages, query errors and graph-reuse details. The [existing examples](../examples/README.md) show broader learning tasks using the native API.